<a href='https://colab.research.google.com/github/fralfaro/ics294-latex/blob/main/talleres/taller_02_solucion.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

# Solución Taller 2: Estimación e informe final

Este notebook completa el proyecto aplicado con `wage1`: estimación MCO, bondad de ajuste, inferencia individual y conjunta, diagnóstico, errores robustos, predicción y conclusiones. En Google Colab, selecciona **Runtime > Change runtime type > R**.

Cada pregunta del Taller 2 aparece identificada y se responde con código, resultados numéricos e interpretación económica.

## 1. Preparación de datos y modelos

In [ ]:
paquetes <- c('wooldridge', 'dplyr', 'ggplot2', 'lmtest', 'sandwich', 'car')
instalar <- paquetes[!vapply(paquetes, requireNamespace, logical(1), quietly = TRUE)]
if (length(instalar) > 0) install.packages(instalar, repos = 'https://cloud.r-project.org')
invisible(lapply(paquetes, library, character.only = TRUE))
theme_set(theme_minimal(base_size = 12))
options(width = 110)

data('wage1', package = 'wooldridge')
wage_df <- as_tibble(wage1)
if (!'lwage' %in% names(wage_df)) wage_df <- wage_df |> mutate(lwage = log(wage))

modelo_uni <- lm(lwage ~ educ, data = wage_df)
modelo_bi <- lm(lwage ~ educ + exper, data = wage_df)
modelo_multi <- lm(lwage ~ educ + exper + tenure, data = wage_df)

## Preguntas 1 y 2. Estimación MCO y bondad de ajuste

In [ ]:
tabla_coeficientes <- function(mod, nombre) {
  co <- as.data.frame(coef(summary(mod)))
  tibble(modelo = nombre, parametro = rownames(co), estimacion = co[,1],
         error_estandar = co[,2], estadistico_t = co[,3], p_value = co[,4])
}

coeficientes <- bind_rows(
  tabla_coeficientes(modelo_uni, 'lwage ~ educ'),
  tabla_coeficientes(modelo_bi, 'lwage ~ educ + exper'),
  tabla_coeficientes(modelo_multi, 'lwage ~ educ + exper + tenure')
) |> mutate(across(where(is.numeric), ~ round(.x, 5)))
coeficientes

tabla_ajuste <- bind_rows(lapply(list(modelo_uni, modelo_bi, modelo_multi), function(mod) {
  s <- summary(mod)
  tibble(modelo = deparse(formula(mod)), n = nobs(mod), R2 = s$r.squared,
         R2_ajustado = s$adj.r.squared, RMSE = s$sigma,
         F_global = unname(s$fstatistic[1]), gl1 = unname(s$fstatistic[2]),
         gl2 = unname(s$fstatistic[3]))
})) |> mutate(across(where(is.numeric), ~ round(.x, 4)))
tabla_ajuste

### Respuesta a la Pregunta 1

El Modelo 3 estimado es aproximadamente:

$$\widehat{lwage}=0{,}2844+0{,}0920\,educ+0{,}0041\,exper+0{,}0221\,tenure.$$

Manteniendo constantes las demás variables, un año adicional de educación se asocia aproximadamente con un aumento de 9,20\% en el salario por hora; la transformación exacta es 9,64\%. Los efectos aproximados de experiencia y antigüedad son 0,41\% y 2,21\%, respectivamente.

Los coeficientes de `educ` son 0,0827 en el Modelo 1, 0,0979 en el Modelo 2 y 0,0920 en el Modelo 3. El cambio refleja que el efecto de educación se estima primero sin controles y luego como asociación parcial, manteniendo constantes experiencia y antigüedad.

### Respuesta a la Pregunta 2

| Modelo | $R^2$ | $R^2$ ajustado | RMSE |
|---|---:|---:|---:|
| `lwage ~ educ` | 0,1858 | 0,1843 | 0,4801 |
| `lwage ~ educ + exper` | 0,2493 | 0,2465 | 0,4614 |
| `lwage ~ educ + exper + tenure` | 0,3160 | 0,3121 | 0,4409 |

El $R^2$ aumenta al agregar variables y el $R^2$ ajustado también mejora. Los estadísticos $F$ globales son aproximadamente 119,58, 86,86 y 80,39 para los tres modelos, respectivamente; todos tienen valor-$p$ menor que 0,001. Esto indica significancia global de cada modelo, pero no demuestra causalidad. El gráfico observado-ajustado debe mostrar puntos alrededor de la diagonal si el ajuste es razonable.

La respuesta a la Pregunta 2 es, por tanto, que el Modelo 3 presenta el mejor ajuste muestral entre las tres especificaciones. La elección final debe considerar también teoría económica, parsimonia y diagnóstico de supuestos.

In [ ]:
ggplot(wage_df, aes(educ, lwage)) + geom_point(alpha = .45, color = '#377eb8') +
  geom_smooth(method = 'lm', se = TRUE, color = '#e41a1c') +
  labs(title = 'Recta estimada: lwage según educación', x = 'educ', y = 'lwage')

observado_ajustado <- tibble(observado = wage_df$lwage, ajustado = fitted(modelo_multi))
ggplot(observado_ajustado, aes(observado, ajustado)) + geom_point(alpha = .5) +
  geom_abline(slope = 1, intercept = 0, linetype = 'dashed', color = '#e41a1c') +
  labs(title = 'Valores observados y ajustados', x = 'lwage observado', y = 'lwage ajustado')

## Pregunta 3. Inferencia individual y errores robustos

In [ ]:
summary(modelo_multi)
confint(modelo_multi, level = .95)

tabla_robusta <- as.data.frame(unclass(coeftest(
  modelo_multi, vcov. = vcovHC(modelo_multi, type = 'HC1')
)))
tabla_robusta$parametro <- rownames(tabla_robusta)
rownames(tabla_robusta) <- NULL
tabla_robusta

### Respuesta a la Pregunta 3

Para cada variable se contrasta $H_0:\beta_j=0$ frente a $H_1:\beta_j\neq0$. Con errores estándar convencionales, los estadísticos son aproximadamente: `educ` $t=12,56$, `exper` $t=2,39$ y `tenure` $t=7,13$. Se rechaza $H_0$ para las tres variables al 5\%.

Los intervalos al 95\% son aproximadamente $[0,0776;0,1064]$ para `educ`, $[0,0007;0,0075]$ para `exper` y $[0,0160;0,0281]$ para `tenure`.

Con errores HC1 robustos, los estadísticos son 11,62, 2,36 y 5,83, con valores-$p$ aproximadamente menores que 0,05. Las conclusiones no cambian: las tres variables siguen siendo significativas al 5\%. Los errores robustos no cambian los coeficientes, sino la estimación de su varianza.

## Preguntas 4 y 5. Restricciones lineales y prueba conjunta

In [ ]:
test_diferencia <- linearHypothesis(modelo_multi, 'educ - exper = 0')
test_diferencia

test_conjunto <- linearHypothesis(modelo_multi, c('exper = 0', 'tenure = 0'))
test_conjunto

modelo_restringido <- lm(lwage ~ educ, data = wage_df)
anova(modelo_restringido, modelo_multi)

### Respuesta a la Pregunta 4

Para $H_0:\beta_1-\beta_2=0$, el estadístico es aproximadamente $F=158,46$ y el valor-$p$ es menor que 0,001. Se rechaza la igualdad: el retorno estimado de educación es mayor que el de experiencia.

Para $H_0:\beta_2=0$ y $\beta_3=0$, el estadístico conjunto es aproximadamente $F=49,69$, también con valor-$p<0,001$. Se rechaza la irrelevancia conjunta de experiencia y antigüedad.

### Respuesta a la Pregunta 5

En la hipótesis conjunta, el Modelo 3 es el no restringido: contiene `educ`, `exper` y `tenure`. El modelo restringido es `lwage ~ educ`, porque impone simultáneamente $\beta_2=0$ y $\beta_3=0$. La estadística $F$ compara el aumento en la suma de residuos que se produce al imponer ambas restricciones, ajustado por sus grados de libertad. No corresponde sustituir esta prueba por dos pruebas $t$ independientes.

El Modelo 3 es el no restringido y `lwage ~ educ` es el restringido. `anova()` entrega la misma conclusión porque las restricciones corresponden a eliminar `exper` y `tenure`.

## Preguntas 6, 7 y 8. Diagnóstico y correcciones

In [ ]:
diagnostico <- tibble(ajustado = fitted(modelo_multi), residuo = resid(modelo_multi),
                      residuo_estandarizado = rstandard(modelo_multi))

ggplot(diagnostico, aes(ajustado, residuo)) + geom_point(alpha = .5) +
  geom_hline(yintercept = 0, linetype = 'dashed', color = '#e41a1c') +
  geom_smooth(method = 'loess', se = FALSE, color = '#4daf4a') +
  labs(title = 'Residuos versus ajustados', x = 'Ajustados', y = 'Residuos')

ggplot(diagnostico, aes(sample = residuo_estandarizado)) + stat_qq(alpha = .5) +
  stat_qq_line(color = '#e41a1c') +
  labs(title = 'QQ plot de residuos', x = 'Cuantiles teóricos', y = 'Cuantiles muestrales')

bp <- bptest(modelo_multi)
bp

# White: regresión auxiliar con niveles, cuadrados e interacciones.
aux_white <- lm(I(resid(modelo_multi)^2) ~ educ + exper + tenure +
  I(educ^2) + I(exper^2) + I(tenure^2) +
  educ:exper + educ:tenure + exper:tenure, data = wage_df)
white_stat <- nrow(wage_df) * summary(aux_white)$r.squared
white_df <- length(coef(aux_white)) - 1
white_p <- pchisq(white_stat, df = white_df, lower.tail = FALSE)
tibble(test = 'White', estadistico = white_stat, gl = white_df, p_value = white_p)

# Jarque--Bera calculado a partir de asimetría y curtosis.
errores <- resid(modelo_multi)
asimetria <- mean((errores - mean(errores))^3) / sd(errores)^3
curtosis <- mean((errores - mean(errores))^4) / sd(errores)^4
jb_stat <- length(errores) / 6 * (asimetria^2 + (curtosis - 3)^2 / 4)
jb_p <- pchisq(jb_stat, df = 2, lower.tail = FALSE)
tibble(test = 'Jarque-Bera', estadistico = jb_stat, gl = 2, p_value = jb_p)

vif(modelo_multi)

### Respuesta a la Pregunta 6

La prueba de Breusch--Pagan entrega aproximadamente $BP=10,76$ y valor-$p=0,013$. La prueba de White completa, usando niveles, cuadrados e interacciones, entrega aproximadamente $LM=20,74$, 9 grados de libertad y valor-$p=0,014$. Ambas rechazan homocedasticidad al 5\%, por lo que se deben reportar errores estándar robustos HC1.

### Respuesta a la Pregunta 7

La prueba de Jarque--Bera entrega aproximadamente $JB=20,30$ y valor-$p<0,001$, por lo que se rechaza normalidad exacta de los residuos. Con 526 observaciones, la inferencia asintótica y los errores robustos son más importantes que asumir normalidad exacta. Los VIF son aproximadamente 1,11 para `educ`, 1,48 para `exper` y 1,35 para `tenure`; no sugieren multicolinealidad preocupante.

Los gráficos permiten revisar patrones de residuos y normalidad, pero no prueban exogeneidad. Variables omitidas, simultaneidad y error de medición pueden seguir afectando la interpretación causal.

En particular, un embudo en el gráfico de residuos sugeriría varianza no constante; una curva sistemática sugeriría que la relación lineal puede ser insuficiente. Un QQ plot alejado de la línea indicaría desviaciones de normalidad, especialmente en las colas.

## Pregunta 8. Aplicación de correcciones

Como Breusch--Pagan y White rechazan homocedasticidad, la corrección principal es reportar errores estándar HC1. Los coeficientes MCO se mantienen; cambian los errores estándar, estadísticos y valores-$p$.

El siguiente bloque estima además una especificación con experiencia cuadrática. Esta modificación solo debe preferirse si existe una justificación económica o evidencia de no linealidad; no se agrega automáticamente para aumentar el $R^2$.

In [ ]:
coeftest(modelo_multi, vcov. = vcovHC(modelo_multi, type = 'HC1'))

modelo_cuadratico <- lm(lwage ~ educ + exper + I(exper^2) + tenure, data = wage_df)
summary(modelo_cuadratico)
coeftest(modelo_cuadratico, vcov. = vcovHC(modelo_cuadratico, type = 'HC1'))

### Respuesta a la Pregunta 8

Con HC1, los coeficientes permanecen iguales, pero se corrigen los errores estándar y los valores-$p$ para la heterocedasticidad. En la especificación cuadrática se obtiene aproximadamente:

$$\widehat{lwage}=0{,}1983+0{,}0853\,educ+0{,}0329\,exper-0{,}00066\,exper^2+0{,}0208\,tenure.$$

El coeficiente de `exper^2` es negativo y estadísticamente significativo, lo que respalda retornos marginales decrecientes de la experiencia. El $R^2$ ajustado sube aproximadamente a 0,3545. Esta especificación es una mejora solo si la curvatura tiene sentido económico y no se justifica únicamente por aumentar el ajuste.

## Pregunta 9. Predicción

In [ ]:
nuevos_trabajadores <- tibble(educ = c(12, 16, 18), exper = 10, tenure = 5)
pred_log <- predict(modelo_multi, newdata = nuevos_trabajadores,
                    interval = 'confidence', level = .95)
pred_salario <- exp(pred_log)
bind_cols(nuevos_trabajadores, as_tibble(pred_log))
bind_cols(nuevos_trabajadores, as_tibble(pred_salario))

### Resultado

| Perfil | Salario predicho | IC 95\% para la media |
|---|---:|---:|
| 12 años de educación | 4,67 | [4,45; 4,89] |
| 16 años de educación | 6,74 | [6,34; 7,16] |
| 18 años de educación | 8,10 | [7,46; 8,81] |

Estos son intervalos de confianza para el salario medio condicional, no intervalos de predicción para trabajadores individuales.

### Tabla consolidada para el informe

La siguiente tabla permite reportar en un solo lugar los coeficientes MCO y la inferencia robusta del modelo principal.

In [ ]:
rob <- as.data.frame(unclass(coeftest(modelo_multi, vcov. = vcovHC(modelo_multi, type = 'HC1'))))
tabla_reporte <- tibble(
  parametro = rownames(rob), estimacion = rob[, 1],
  error_estandar_HC1 = rob[, 2], estadistico_t = rob[, 3],
  p_value = rob[, 4]
) |> mutate(across(where(is.numeric), ~ round(.x, 5)))
tabla_reporte

## Pregunta 10. Conclusión para el informe

En `wage1`, educación, experiencia y antigüedad presentan asociaciones positivas con `lwage` en el Modelo 3. Las tres variables son significativas al 5\%, incluso al utilizar errores robustos. Experiencia y antigüedad son conjuntamente significativas, y el retorno estimado de educación es mayor que el de experiencia.

El Modelo 3 explica aproximadamente 31,6\% de la variación muestral de `lwage`. Las pruebas de Breusch--Pagan y White sugieren heterocedasticidad, mientras que los VIF no sugieren multicolinealidad importante.

La conclusión debe ser prudente: el análisis es observacional y, sin una estrategia adicional de identificación, los coeficientes deben interpretarse como asociaciones condicionadas y no como efectos causales.

### Recomendaciones

Se recomienda reportar errores estándar robustos, mantener controles económicamente justificados, revisar especificaciones no lineales de experiencia y realizar análisis de sensibilidad frente a observaciones influyentes. Para sostener afirmaciones causales se necesitaría una estrategia adicional, como variables instrumentales, datos longitudinales o un diseño cuasiexperimental.

### Referencias sugeridas

Wooldridge, J. M. (2019). *Introductory Econometrics: A Modern Approach*. Cengage Learning. La base `wage1` se utiliza a través del paquete `wooldridge`.